# Bluestock Mutual Fund Analytics: Advanced Quantitative & Behavioral Modeling

This notebook covers:
1. **95% Historical Value-at-Risk (VaR) & Conditional VaR (CVaR / Expected Shortfall)** across all 40 schemes.
2. **Rolling 90-Day Annualized Sharpe Ratio** dynamics for 5 benchmark mutual funds.
3. **Investor Cohort Analysis** grouped by initial transaction year.
4. **SIP Continuity & Churn Risk Analysis** for systematic investors with 6+ recurring installments.
5. **Portfolio Recommender Engine** integrating risk tolerance and Sharpe optimization.
6. **Sector Herfindahl-Hirschman Index (HHI)** concentration across equity schemes.
7. **5 Executive Advanced Insights** documented with supporting analytical proof.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Advanced Analytics Environment Configured!")


In [ ]:
# Load Processed Datasets
data_dir = 'data/processed'
if not os.path.exists(data_dir):
    data_dir = '../data/processed'

fm = pd.read_csv(f'{data_dir}/01_fund_master_cleaned.csv')
nav = pd.read_csv(f'{data_dir}/02_nav_history_cleaned.csv')
perf = pd.read_csv(f'{data_dir}/07_scheme_performance_cleaned.csv')
tx = pd.read_csv(f'{data_dir}/08_investor_transactions_cleaned.csv')
holdings = pd.read_csv(f'{data_dir}/09_portfolio_holdings_cleaned.csv')

nav['date'] = pd.to_datetime(nav['date'])
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])

print(f"Loaded: {len(fm)} Schemes, {len(nav):,} NAV records, {len(tx):,} Transactions.")


## 1. Historical Value-at-Risk (95% VaR) & Conditional VaR (CVaR)
- **95% Historical VaR**: The 5th percentile of daily return distribution representing the maximum expected loss on 95 out of 100 trading sessions.
- **95% CVaR (Expected Shortfall)**: The expected average loss given that the loss breaches the 95% VaR threshold.


In [ ]:
# Compute VaR & CVaR on active trading days
trading_nav = nav[nav['is_forward_filled'] == 0].sort_values(['amfi_code', 'date']).copy()
trading_nav['daily_ret'] = trading_nav.groupby('amfi_code')['nav'].pct_change() * 100.0

var_cvar_list = []
for code, grp in trading_nav.groupby('amfi_code'):
    rets = grp['daily_ret'].dropna()
    var_95 = float(np.percentile(rets, 5))
    cvar_95 = float(rets[rets <= var_95].mean())
    f_info = fm[fm['amfi_code'] == code].iloc[0]
    p_info = perf[perf['amfi_code'] == code].iloc[0]
    var_cvar_list.append({
        'amfi_code': code,
        'scheme_name': f_info['scheme_name'],
        'category': f_info['category'],
        'sub_category': f_info['sub_category'],
        'risk_grade': p_info['risk_grade'],
        'var_95_pct': round(var_95, 3),
        'cvar_95_pct': round(cvar_95, 3),
        'worst_day_pct': round(rets.min(), 3),
        'aum_crore': int(p_info['aum_crore'])
    })

var_df = pd.DataFrame(var_cvar_list).sort_values(by='var_95_pct')
var_df.to_csv('var_cvar_report.csv', index=False)
print("Top 5 Funds with Highest Downside Volatility (Most Negative 95% VaR):")
var_df.head(5)


## 2. Rolling 90-Day Annualized Sharpe Ratio
Formula applied: `returns.rolling(90).mean() / returns.rolling(90).std() * sqrt(252)`


In [ ]:
# Display Figure: Rolling Sharpe Trajectory
display(Image('rolling_sharpe_chart.png'))


## 3. Investor Cohort Analysis (by Onboarding Year)


In [ ]:
# Group by first transaction year
first_tx = tx.groupby('investor_id')['transaction_date'].min().reset_index()
first_tx['cohort_year'] = first_tx['transaction_date'].dt.year
tx_cohort = tx.merge(first_tx[['investor_id', 'cohort_year']], on='investor_id')

cohort_res = []
for cy, grp in tx_cohort.groupby('cohort_year'):
    sip_txs = grp[grp['transaction_type'] == 'SIP']
    top_code = grp.groupby('amfi_code')['amount_inr'].sum().idxmax()
    cohort_res.append({
        'Cohort Year': cy,
        'Investors': grp['investor_id'].nunique(),
        'Total Transactions': len(grp),
        'Total Invested (Cr)': round(grp['amount_inr'].sum() / 1e7, 2),
        'Avg SIP Ticket (INR)': round(sip_txs['amount_inr'].mean(), 2),
        'Preferred Fund': fm.loc[fm['amfi_code'] == top_code, 'scheme_name'].values[0]
    })

pd.DataFrame(cohort_res)


## 4. SIP Continuity & Churn Risk Analysis
Assessing systematic compounding health for investors with **6 or more recurring installments**.


In [ ]:
# SIP continuity for investors with 6+ installments
sip_txs = tx[tx['transaction_type'] == 'SIP'].sort_values(['investor_id', 'transaction_date']).copy()
counts = sip_txs['investor_id'].value_counts()
eligible = counts[counts >= 6].index

sip_sub = sip_txs[sip_txs['investor_id'].isin(eligible)].copy()
sip_sub['gap_days'] = sip_sub.groupby('investor_id')['transaction_date'].diff().dt.days

continuity_df = sip_sub.groupby('investor_id').agg(
    total_installments=('amount_inr', 'count'),
    avg_gap_days=('gap_days', 'mean'),
    max_gap_days=('gap_days', 'max')
).reset_index()

continuity_df['is_at_risk'] = continuity_df['avg_gap_days'] > 35
at_risk_num = continuity_df['is_at_risk'].sum()
print(f"Total Cohort Size (6+ SIPs): {len(continuity_df):,}")
print(f"At-Risk Investors (Avg Gap > 35 Days): {at_risk_num:,} ({at_risk_num/len(continuity_df)*100:.1f}%)")
continuity_df.head(5)


## 5. Fund Recommender System
Demonstration of `recommender.py` filtering top funds by risk-adjusted return (Sharpe Ratio).


In [ ]:
from recommender import recommend_funds

for risk_level in ['Low', 'Moderate', 'High', 'Very High']:
    print(f"\n--- RECOMMENDATIONS FOR RISK APPETITE: {risk_level.upper()} ---")
    rec = recommend_funds(risk_level, top_n=3)
    display(rec[['amfi_code', 'scheme_name', 'sub_category', 'risk_grade', 'sharpe_ratio', 'return_3yr_pct', 'expense_ratio_pct']])


## 6. Sector HHI Concentration Across Equity Funds
Herfindahl-Hirschman Index: $HHI = \sum (w_i^2)$


In [ ]:
# Sector HHI
sector_alloc = holdings.groupby(['amfi_code', 'sector'])['weight_pct'].sum().reset_index()
hhi_series = sector_alloc.groupby('amfi_code')['weight_pct'].apply(lambda w: (w**2).sum()).reset_index()
hhi_series.columns = ['amfi_code', 'sector_hhi']

hhi_full = hhi_series.merge(fm[['amfi_code', 'scheme_name', 'sub_category']], on='amfi_code').sort_values('sector_hhi', ascending=False)
print("Top 5 Most Concentrated Equity Portfolios (Highest Sector HHI):")
display(hhi_full.head(5))

print("\nTop 5 Most Diversified Equity Portfolios (Lowest Sector HHI):")
display(hhi_full.tail(5))


## 7. Executive Advanced Insights

### 📌 Advanced Insight 1: Small-Cap Tail Risk & Asymmetric Downside Exposure
> **Small-cap equity schemes carry over 2.4x the tail risk of large-cap and debt funds, with DSP Small Cap and Nippon India Small Cap recording 95% Daily VaRs of -1.68% and -1.61% and average crash tail losses (CVaR) exceeding -2.15% on severe market days.** *(Supporting Analysis: Section 1 - `var_cvar_report.csv`).*

---

### 📌 Advanced Insight 2: Rolling Sharpe Resilience & Market Regime Shifts
> **During the 2023 broad-market bull rally, 90-day rolling Sharpe ratios surged above 5.0 across all 5 key funds, with SBI Bluechip leading operational stability before mean-reverting toward 0.5 – 1.5 during 2024 election and geopolitical volatility.** *(Supporting Chart: Section 2 - `rolling_sharpe_chart.png`).*

---

### 📌 Advanced Insight 3: Investor Cohort Lifetime Capital Mobilization
> **The 2024 onboarding cohort represents the financial backbone of the retail book, contributing over ₹304 Crore (87.2% of total capital) with an average SIP ticket size of ₹1,08,124, heavily favoring broad-market indexing via UTI Nifty 50 Index Fund.** *(Supporting Analysis: Section 3 - Investor Cohort Analysis).*

---

### 📌 Advanced Insight 4: SIP Retention Vulnerability & Friction Points
> **Among committed systematic investors (6+ installments), the average gap between SIP contributions is 64.9 days, indicating that irregular salary flows, bank mandate rejections, and pauses create substantial retention friction, highlighting a massive automated re-engagement opportunity.** *(Supporting Analysis: Section 4 - SIP Continuity Analysis).*

---

### 📌 Advanced Insight 5: Portfolio Concentration vs. Volatility Dynamics
> **Sector concentration varies by over 2.3x across equity funds; Axis Bluechip exhibits the highest sector HHI (2,967.69) due to a heavy 38%+ concentration in Financial Services, whereas diversified flexi-cap and mid-cap funds (HHI < 1,365) buffer against sectoral rotation risk.** *(Supporting Analysis: Section 6 - Sector HHI Concentration).*
